# Model Deployment Testing Notebook

This notebook mirrors the new model-focused inference path used by `app/model_streamlit_app.py`.

It is intentionally separate from the original exploratory-analysis deployment path.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.inference_utils import (
    build_debug_summary,
    build_inference_frame,
    build_inference_ready_preview,
    build_manual_defaults,
    load_feature_importance,
    load_model_bundle,
    load_reference_dataset,
    predict_record,
    sample_reference_row,
)
from app.model_metadata import NOTEBOOK_METRICS, PROJECT_NAME


## Load the Saved Model and Supporting Files

The saved joblib artifact is the source of truth for inference. The cleaned dataset and feature-importance CSV are used for schema support and interpretation.

In [ ]:
bundle = load_model_bundle()
reference_df = load_reference_dataset()
feature_importance_df = load_feature_importance()

pd.DataFrame(
    [
        {"item": "project", "value": PROJECT_NAME},
        {"item": "saved_model_artifact", "value": str(bundle["artifact_path"])},
        {"item": "selected_model", "value": bundle.get("model_name")},
        {"item": "selected_config", "value": bundle.get("selected_config")},
        {"item": "test_roc_auc", "value": NOTEBOOK_METRICS["test_roc_auc"]},
        {"item": "reference_rows", "value": len(reference_df)},
        {"item": "feature_importance_rows", "value": len(feature_importance_df)},
    ]
)


In [ ]:
build_debug_summary(bundle)


## Manual Inference Example

This example uses the same helper functions as the Streamlit app. The helper computes the engineered fields (`cust-loyality`, `family_member`, `subscription_count`) before prediction.

In [ ]:
defaults = build_manual_defaults(reference_df)

manual_inputs = {
    **defaults,
    "tenure": 6,
    "partner": "No",
    "dependents": "No",
    "phoneservice": "Yes",
    "multiplelines": "No",
    "internetservice": "Fiber optic",
    "onlinesecurity": "No",
    "onlinebackup": "No",
    "deviceprotection": "No",
    "techsupport": "No",
    "streamingtv": "Yes",
    "streamingmovies": "Yes",
    "contract": "Month-to-month",
    "paperlessbilling": "Yes",
    "paymentmethod": "Electronic check",
    "monthlycharges": 95.0,
    "totalcharges": 570.0,
}

manual_preview = build_inference_ready_preview(manual_inputs)
manual_preview


In [ ]:
manual_frame = build_inference_frame(manual_inputs)
manual_result = predict_record(bundle, manual_frame)

pd.DataFrame(
    [
        {
            "predicted_class": manual_result["predicted_label"],
            "yes_probability": manual_result.get("yes_probability"),
            "predicted_probability": manual_result.get("predicted_probability"),
            "decision_score": manual_result.get("decision_score"),
            "summary": manual_result["summary_sentence"],
        }
    ]
)


## Auto-Generated Sample Checks

These samples are drawn from real historical rows in `cleaned_data.csv`, which makes them useful for sanity-checking both classes quickly.

In [ ]:
comparison_rows = []
for intended_class, seed in [("No", 1), ("Yes", 2)]:
    sample = sample_reference_row(reference_df, intended_class=intended_class, random_state=seed)
    prediction = predict_record(bundle, sample["model_frame"])
    comparison_rows.append(
        {
            "intended_class": intended_class,
            "predicted_class": prediction["predicted_label"],
            "match": prediction["predicted_label"] == intended_class,
            "yes_probability": prediction.get("yes_probability"),
        }
    )

pd.DataFrame(comparison_rows)


## Notes

- The old `telecom_deployment.py` script remains the exploratory-analysis dashboard.
- This notebook and the new Streamlit app are the new model-only deployment/testing path.
- The saved artifact already contains the fitted preprocessing pipeline, so inference reuses the exact training pipeline.